# Stage 0 — 평지 Goal 도달 학습 (Colab)

**담당**: 이재왕 (work/evader)  
**씬**: `Assets/00. LJW/Scenes/Stage0_Flat.unity`  
**목표**: 장애물·Pursuer 없는 평지에서 드론이 GoalZone에 도달하도록 학습  
**수렴 기준**: `goal_reach_rate ≥ 50%` → Stage0-B (Pursuer 추가)로 전환

---

## Unity 씬 설정 (로컬에서 한 번 수행)

### 1. 씬 생성
```
File > New Scene → 저장 위치: Assets/00. LJW/Scenes/Stage0_Flat.unity
```

### 2. Hierarchy 구성
```
Stage0_Flat
├── Ground       (GameObject > 3D Object > Plane, Scale = 20,1,20)
├── EvaderDrone  (빈 GameObject — 드론 Rigidbody + 컴포넌트 추가)
│   ├── Rigidbody        (Mass=1, Angular Drag=2)
│   ├── DronePhysics     (Assets/02. Scripts/DronePhysics.cs 추가)
│   ├── EvaderAgent      (Assets/00. LJW/Scripts/EvaderAgent.cs 추가)
│   ├── EvaderReward     (Assets/00. LJW/Scripts/EvaderReward.cs 추가)
│   ├── BehaviorParameters
│   └── DecisionRequester
└── GoalZone     (빈 GameObject — SphereCollider 추가)
```

### 3. 컴포넌트 설정값

| 컴포넌트 | 파라미터 | 값 |
|---|---|---|
| DronePhysics | ThrustForce | 10 |
| DronePhysics | MaxSpeed | 8 |
| DronePhysics | MaxAltitude | 30 |
| EvaderAgent | _goalOnlyMode | ✅ true |
| EvaderAgent | _goalTransform | GoalZone 연결 |
| EvaderAgent | _pursuerTransform | **(비움)** |
| EvaderAgent | _dronePhysics | DronePhysics 연결 |
| EvaderAgent | _spawnAltitude | 5 |
| EvaderAgent | _spawnRadius | 10 |
| EvaderAgent | _goalRandomizeRadius | 20 |
| BehaviorParameters | Behavior Name | **EvaderAgent** |
| BehaviorParameters | Space Type | Continuous |
| BehaviorParameters | Continuous Actions | **4** |
| BehaviorParameters | Vector Obs Size | **18** |
| DecisionRequester | Decision Period | **5** |
| GoalZone | SphereCollider Is Trigger | ✅ |
| GoalZone | SphereCollider Radius | 2.0 |
| GoalZone | Tag | **goal** |

> ⚠️ BehaviorParameters Behavior Name이 **EvaderAgent** 여야 YAML 설정과 매핑됩니다.

### 4. Linux 빌드
```
File > Build Settings
  Platform: Linux (x86_64)
  ✅ Server Build  ← 헤드리스 모드 필수
  Add Open Scenes → Stage0_Flat 씬만 포함
  Build → 파일명: EvaderEnv
```
빌드 폴더 `EvaderEnv/` 를 Google Drive `IIT_DroneLearning/builds/EvaderEnv/` 에 업로드

---

## 실행 전 체크리스트
- [ ] GPU 런타임 활성화 (런타임 > 런타임 유형 변경 > T4 GPU)
- [ ] Google Drive에 Linux Headless 빌드 업로드 완료
- [ ] `DRIVE_BASE` 경로를 실제 Drive 경로로 수정

---
## 0. Colab 연결 유지 (방치 시 브라우저 Console에 붙여넣기)

In [ ]:
# JavaScript를 브라우저 개발자 도구 Console에 붙여넣으면 자동 연결 해제 방지:
# function ClickConnect(){console.log('keep alive'); document.querySelector('#top-toolbar > colab-connect-button').shadowRoot.querySelector('#connect').click();} setInterval(ClickConnect, 60000);
print('위 JavaScript를 브라우저 Console에 붙여넣으면 자동 연결 해제를 방지할 수 있습니다.')

---
## 1. 환경 설치

In [ ]:
import sys

# 1. pettingzoo==1.15.0 은 PyPI에 없음 → 호환 버전(1.22.3)으로 선설치
#    mlagents-envs release_23이 pettingzoo==1.15.0 을 pin하지만 API 호환됨
!{sys.executable} -m pip install -q "pettingzoo==1.22.3"

# 2. mlagents-envs: --no-deps 로 pettingzoo 버전 핀 우회
!{sys.executable} -m pip install -q --no-deps \
    "mlagents-envs @ git+https://github.com/Unity-Technologies/ml-agents.git@release_23#subdirectory=ml-agents-envs"

# 3. mlagents: --no-deps 로 mlagents_envs==1.2.0.dev0 (PyPI 없음) 우회
!{sys.executable} -m pip install -q --no-deps \
    "mlagents @ git+https://github.com/Unity-Technologies/ml-agents.git@release_23#subdirectory=ml-agents"

# 4. 나머지 의존성 호환 버전으로 수동 설치
#    grpcio>1.53 / protobuf>=4 / numpy>=2 는 mlagents release_23과 충돌
!{sys.executable} -m pip install -q \
    "grpcio>=1.11.0,<=1.53.2" \
    "protobuf>=3.20.0,<4.0" \
    "numpy>=1.21.0,<2.0" \
    "h5py>=2.9.0" \
    "Pillow>=4.2.1" \
    "PyYAML>=5.1" \
    "cattrs>=1.1.0" \
    "attrs>=21.0.0"

# 5. PyTorch 및 학습 도구
!{sys.executable} -m pip install -q "torch>=2.0.0,<3.0.0" tensorboard onnx

# 설치 확인
import importlib; importlib.invalidate_caches()
import mlagents, torch
print(f'mlagents : {mlagents.__version__}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'Python   : {sys.version}')

---
## 2. Google Drive 마운트 및 경로/실험 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── 경로 설정 ────────────────────────────────────────────────────────────
DRIVE_BASE      = '/content/drive/MyDrive/IIT_DroneLearning'  # ← 실제 경로로 수정
REPO_PATH       = '/content/IIT_DroneLearning'
BUILD_PATH      = f'{DRIVE_BASE}/builds/EvaderEnv'
LOG_DIR         = f'{DRIVE_BASE}/runs'
CHECKPOINT_DIR  = f'{DRIVE_BASE}/checkpoints'

for d in [LOG_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 실험 설정 ────────────────────────────────────────────────────────────
SEED        = 42
NUM_ENVS    = 4      # Colab Pro+ T4 기준
INIT_FROM   = None   # warm-start 필요 시 이전 run-id 입력

RUN_ID = f'evader_s0_flat_seed{SEED}'

print(f'Run ID  : {RUN_ID}')
print(f'Log dir : {LOG_DIR}')

---
## 3. 레포 클론 및 Config 확인

In [ ]:
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/alpha7179/IIT_DroneLearning.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git fetch origin work/evader && git reset --hard origin/work/evader

!cd {REPO_PATH} && git checkout work/evader && git log --oneline -3

# Config 탐색 — flat 전용 파일 우선, 없으면 base 폴백
config_candidates = [
    f'{REPO_PATH}/python/config/evader_s0_flat_template.yaml',
    f'{REPO_PATH}/python/config/evader_s0_20260315_base.yaml',
    f'{REPO_PATH}/python/config/evader_s0_template.yaml',
]
config_path = next((c for c in config_candidates if os.path.exists(c)), None)
assert config_path, f'Config를 찾을 수 없습니다: {config_candidates}'

print(f'\nConfig: {config_path}')
print('=' * 60)
!cat {config_path}

---
## 4. Unity Linux Headless 빌드 확인

In [ ]:
build_exe = f'{BUILD_PATH}/EvaderEnv.x86_64'

if os.path.exists(build_exe):
    !chmod +x {build_exe}
    print(f'✅ Build found: {build_exe}')
    BUILD_READY = True
else:
    print(f'❌ Build NOT found: {build_exe}')
    print('Unity에서 Stage0_Flat.unity 씬을 Linux Headless 빌드하여 Drive에 업로드하세요.')
    BUILD_READY = False

---
## 5. 학습 실행

In [ ]:
import subprocess

assert BUILD_READY, '빌드 파일이 없습니다. 4번 셀을 먼저 확인하세요.'

cmd = [
    'mlagents-learn', config_path,
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
    f'--env={build_exe}',
    f'--num-envs={NUM_ENVS}',
    '--no-graphics',
    '--force',
]

if INIT_FROM:
    cmd += [f'--initialize-from={str(Path(LOG_DIR) / INIT_FROM)}']
    print(f'Warm-start from: {INIT_FROM}')

print('실행 커맨드:')
print(' '.join(cmd))
print()

result = subprocess.run(cmd, cwd=REPO_PATH)
print(f'\n학습 종료. Exit code: {result.returncode}')

---
## 6. TensorBoard 모니터링 (학습 중 병렬 실행)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

# 확인 지표:
# Environment/Cumulative Reward  : 상승 추세 확인 (> 0.5 안정이면 수렴)
# Environment/Episode Length     : 감소 추세 확인 (더 빨리 Goal 도달)
# Policy/Entropy                 : 초반 높고 서서히 감소
#
# 수렴 기준:
#   Cumulative Reward > 0.5 안정
#   → eval로 goal_reach_rate >= 50% 확인 후 Stage0-B 전환

---
## 7. 체크포인트 Drive 저장 (학습 완료 후)

In [ ]:
import shutil

src = Path(LOG_DIR) / RUN_ID
dst = Path(CHECKPOINT_DIR) / RUN_ID

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'✅ Saved: {dst}')
    onnx_files = list(dst.glob('**/*.onnx'))
    if onnx_files:
        print(f'ONNX ({len(onnx_files)}개):')
        for f in onnx_files:
            print(f'  {f}')
    else:
        print('ONNX 없음 — max_steps 미도달 또는 수동 export 필요')
else:
    print(f'❌ Run 디렉토리 없음: {src}')

---
## 8. Stage0-B 전환 (수렴 확인 후)

수렴 확인 기준:
- TensorBoard `Cumulative Reward > 0.5` 안정
- `goal_reach_rate >= 50%`

전환 방법:

**Unity Inspector 변경:**
- `EvaderAgent._goalOnlyMode` = **false** (체크 해제)
- `EvaderAgent._pursuerTransform` = ScriptedEvader (또는 Pursuer 오브젝트) 연결

**노트북 2번 셀 변경:**
```python
INIT_FROM = 'evader_s0_flat_seed42'  # Stage0-A run-id
RUN_ID    = f'evader_s0b_withpursuer_seed{SEED}'
```

Linux 빌드 재수행 후 5번 셀 재실행.

---
## 9. 실험 결과 기록

학습 완료 후 로컬에서 `docs/EXPERIMENTS.md`에 기록:
```bash
git add docs/EXPERIMENTS.md
git commit -m '[Docs] Stage0 평지 실험 결과 기록'
git push origin work/evader
```